In [10]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from pymilvus import MilvusClient

# Authentication not enabled
client = MilvusClient(
  uri="http://localhost:19530",
  db_name="default",
  user="loc",
  password="loc",

)

DEBUG:pymilvus.milvus_client.milvus_client:Created new connection using: 9e2fe80246dc481f83bf3b5f88498e47


ERROR:pymilvus.milvus_client.milvus_client:Failed to create new connection using: 8bbcaacd905249808613d22ff7e50cb4


TypeError: Connections.connect() got multiple values for argument 'alias'

In [2]:
client

In [2]:
client.drop_collection('test')

In [3]:
client.has_collection('test')

False

In [4]:
client.create_collection(collection_name="test_collection", dimension=5)

In [5]:
from pymilvus import model

embed = model.hybrid.BGEM3EmbeddingFunction(
                model_name='BAAI/bge-m3',
                device='cpu',
                use_fp16=False
            )


/Users/lochoang/Documents/Workspaces/capstone-project/python-server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 86007.60it/s]


In [15]:
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

data =[]

for i, line in enumerate(docs):
	res = embed.encode_documents([line])
	vector = res['dense'][0].tolist()

	data.append({
		"id":i,
		"vector":vector,
		"text":line
	})



In [24]:
embed_dim = len(data[0]['vector'])

In [25]:
embed_dim

1024

In [27]:
client.create_collection(
  collection_name='test_collection_1',
  dimension=embed_dim,
  metric_type="IP", 
  consistency_level="Strong",  
  )

In [1]:
client.create_collection(
  collection_name='test_collection_1',
  dimension=embed_dim,
  metric_type="IP", 
  consistency_level="Strong", 
  id_type='str' 
  )

NameError: name 'client' is not defined

In [28]:
res = client.insert(
	collection_name="test_collection_1",
	data=data
)

In [47]:
question = "When was artificial intelligence founded"
query_embed = embed([question])['dense'][0].tolist()

In [48]:
search_res = client.search(
    collection_name='test_collection_1',
    data=[
        query_embed
    ],  # Use the `emb_text` function to convert the question to an embedding vector
    limit=3,  # Return top 3 results
    search_params={"metric_type": "IP", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
    # filter="distance > 0.2"
)

In [49]:
search_res

data: ["[{'id': 0, 'distance': 0.7777445316314697, 'entity': {'text': 'Artificial intelligence was founded as an academic discipline in 1956.'}}, {'id': 1, 'distance': 0.7777443528175354, 'entity': {'text': ['Artificial intelligence was founded as an academic discipline in 1956.', 'Alan Turing was the first person to conduct substantial research in AI.', 'Born in Maida Vale, London, Turing was raised in southern England.']}}, {'id': 2, 'distance': 0.7777443528175354, 'entity': {'text': ['Artificial intelligence was founded as an academic discipline in 1956.', 'Alan Turing was the first person to conduct substantial research in AI.', 'Born in Maida Vale, London, Turing was raised in southern England.']}}]"] 

In [52]:
client.has_collection('test_collection_1')


False

In [51]:
client.drop_collection('test_collection_1')